In [ ]:
# !pip install GitPython

In [2]:
import git

repo_url = "https://github.com/nikhildhimam25-cmyk/fastapi.git"
repo_dir = "git_incoming_data"
git.Repo.clone_from(repo_url, repo_dir)

<git.repo.base.Repo 'd:\\nikhil\\python\\copilot\\git_incoming_data\\.git'>

In [50]:
from langchain_community.document_loaders import DirectoryLoader , TextLoader
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [9]:
# loader = DirectoryLoader(
#     "D:\\nikhil\\python\\copilot\\git_incoming_data",
#     glob="**/*",      
#     loader_cls=TextLoader,
#     loader_kwargs={"autodetect_encoding": True}
# )

# docs = loader.load()
# print(f"Total files loaded: {len(docs)}")

In [ ]:
import os
from langchain_core.documents import Document 

VALID_EXTENSIONS = (
    ".py", ".js", ".ts", ".jsx", ".tsx", ".html", ".css", ".scss",
    ".dart", ".kt", ".kts", ".java", ".gradle", ".swift", ".m", ".mm",
    ".c", ".cpp", ".cc", ".h", ".hpp", ".rs", ".go", ".rb", ".php",
    ".json", ".yaml", ".yml", ".toml", ".ini", ".sh", ".bash", ".zsh",
    ".ps1", ".sql", ".md", ".txt", ".rst", ".dockerfile", ".gitignore",
)

def load_clean_files(path):
    docs = []

    for root, dirs, files in os.walk(path):
        dirs[:] = [d for d in dirs if d not in [".git", "node_modules", "dist", "build"]]

        for file in files:
            if not file.endswith(VALID_EXTENSIONS):
                continue

            filepath = os.path.join(root, file)
            try:
                with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
                    content = f.read()
                    docs.append(Document(          # ✅ Document object
                        page_content=content,
                        metadata={"source": filepath}  # ✅ metadata with path
                    ))
            except:
                pass

    return docs



In [36]:
docs = load_clean_files("D:\\nikhil\\python\\copilot\\git_incoming_data")

In [38]:
print(len(docs))

7


In [39]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
from pathlib import Path


EXT_MAP = {
    ".py": Language.PYTHON,
    ".js": Language.JS,
    ".jsx": Language.JS,
    ".ts": Language.TS,
    ".tsx": Language.TS,
    ".java": Language.JAVA,
    ".kt": Language.KOTLIN,
    ".kts": Language.KOTLIN,
    ".cpp": Language.CPP,
    ".cc": Language.CPP,
    ".hpp": Language.CPP,
    ".c": Language.C,
    ".h": Language.C,
    ".go": Language.GO,
    ".rs": Language.RUST,
    ".rb": Language.RUBY,
    ".swift": Language.SWIFT,
    ".html": Language.HTML,
    ".sol": Language.SOL,
    ".php": Language.PHP,
    ".scala": Language.SCALA,

}

chunks = []

for doc in docs:
    source = doc.metadata["source"]
    ext = Path(source).suffix

    # skip non-code files
    if ext not in VALID_EXTENSIONS and Path(source).name not in VALID_EXTENSIONS:
        continue

    language = EXT_MAP.get(ext, None)

    if language:
        splitter = RecursiveCharacterTextSplitter.from_language(
            language=language,
            chunk_size=1000,
            chunk_overlap=200
        )
    else:
        # generic for yaml, json, css, dart, sql etc
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200
        )

    chunks.extend(splitter.split_documents([doc]))

print(f"Total chunks: {len(chunks)}")

Total chunks: 8


In [40]:
print(type(docs[1]))

<class 'langchain_core.documents.base.Document'>


In [42]:
from langchain_huggingface import HuggingFaceEmbeddings
emb=HuggingFaceEmbeddings(model="BAAI/bge-base-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [43]:
from langchain_community.vectorstores import FAISS
db=FAISS.from_documents(chunks,emb)

In [45]:
retriever=db.as_retriever(search_type="similarity",search_kwargs={"k":5})

In [46]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
You are an expert code explainer. You will be given code chunks from a GitHub repository.
Your job is to explain the code clearly and in simple terms.

Use the following context (code chunks) to answer the question.

Context:
{context}

Question: {question}

Instructions:
- Explain what the code does in simple terms
- Mention which file it belongs to
- Explain functions, classes, logic step by step
- If multiple files are involved, explain each one
- If you don't know, just say "Not found in the repository"

Answer:
""",
    input_variables=["context", "question"]
)

In [48]:
llm=ChatOllama(model="qwen2.5-coder:7b",temperature=0.1)

In [51]:
rag_chain=({
    "context":retriever,"question":RunnablePassthrough()
}|prompt|llm|StrOutputParser())

In [52]:
query="explain the code "
response=rag_chain.invoke(query)

In [ ]:
query="explain the code "
response=rag_chain.invoke(query)

The provided code snippets belong to a Python project that uses FastAPI and Uvicorn for creating web APIs. The project includes several modules for different functionalities such as predicting heart disease and BMI.

### 1. `app.py`
This file contains the main logic for predicting heart disease using a machine learning model.

#### Functions:
- **`printName()`**: This function returns a JSON response with a message "Hello Baljinder".
- **`printNaam(naam: str)`**: This function takes a name as input and returns a JSON response with the message "Name is [name]".
- **`predictDisease(data: heartInput)`**: This function predicts whether a person has heart disease based on input data. It uses a pre-trained model (`classifier`) to make predictions.

#### Classes:
- **`heartInput(BaseModel)`**: This Pydantic class defines the structure of the input data required for predicting heart disease. It includes fields like age, sex, cholesterol levels, etc.

### 2. `inputData.py`
This file defines the